# SymBro — analyzing pipeline results

Run this **locally**, from the project directory that contains your `.symbro/`
checkpoint folder (the same directory you ran `symbro query`/`rfdiffusion`/`predict`
from) — no GPU, no Colab needed. Needs `symbro` itself installed (`pip install -e .`
from the repo, same as running the CLI — this notebook calls
`pipeline.join_predict_with_pmpnn()` directly rather than reimplementing that join, see
Section 2) plus `pandas`/`matplotlib`, both already SymBro's own base dependencies.
`py3Dmol`/`ipywidgets` are optional, only for the inline 3D preview near the end.

**What this does:** loads `predict.pkl`/`pmpnn.pkl`/`rfdiffusion.pkl` straight off
disk, joins ProteinMPNN's own per-sequence confidence score back onto each validated
candidate (predict.pkl doesn't carry it directly), and gives you a few views on the
result: a self-consistency RMSD vs. pLDDT scatter, a per-assembly/symmetry-type
breakdown, an adjustable re-thresholding view for building a tighter shortlist than
whatever `--max-rmsd`/`--min-plddt` you originally ran `symbro predict` with, and an
export of that shortlist (table + structures) to a folder.

**One real limitation, worth knowing up front:** `predict.pkl` only ever contains
candidates that *passed* self-consistency screening — `boltz.run()`/`alphafold2.run()`/
`af3.run()` compute RMSD/pLDDT for every shortlisted candidate internally, but only
`select_validated_designs()`'s *winners* get saved to the checkpoint (see
`selfconsistency.py`). So this notebook can show you the spread *among your validated
designs*, and let you re-threshold more strictly within that set, but it can't show you
the full attempted distribution including near-misses — that would need a change to
`toolkit/selfconsistency.py` to also persist the unfiltered results, which is out of
scope for this first pass.

**Standard metrics used here**, for reference (see the RFdiffusion paper, Watson et al.
2023, *Nature*, and `dauparas/ProteinMPNN`'s own author notes):
- **`rmsd_to_design`** — self-consistency RMSD: backbone RMSD between the RFdiffusion
  design and the refolded structure. The RFdiffusion paper's own success threshold is
  **&lt;2 Å**.
- **`mean_plddt`** — the refolded structure's mean predicted confidence (0–100).
  RFdiffusion's paper uses **&gt;80** as a filtering threshold.
- **`global_score`** (joined in from `pmpnn.pkl`) — ProteinMPNN's own confidence in the
  sequence given the backbone (negative log-likelihood; lower is better). Useful as a
  secondary, free signal, but per ProteinMPNN's own author: not something to rank final
  designs by on its own.

## 1. Load checkpoints

Reads straight off `.symbro/*.pkl` in the current directory — the authoritative
checkpoint files every `symbro` command itself reads/writes (see `pipeline.py`'s own
`save_checkpoint()`/`load_checkpoint()`).

In [ ]:
import os

import pandas as pd

from toolkit import pipeline

STATE_DIR = ".symbro"


def load_checkpoint(stage):
    path = os.path.join(STATE_DIR, f"{stage}.pkl")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No {path} found. Run this notebook from the project directory containing "
            f".symbro/ (the same directory you ran `symbro {stage}` from), and make sure "
            f"you've actually run that stage yet."
        )
    return pd.read_pickle(path)


predict_df = load_checkpoint("predict")
pmpnn_df = load_checkpoint("pmpnn")
rfdiffusion_df = load_checkpoint("rfdiffusion")

if predict_df.empty:
    raise RuntimeError(
        "predict.pkl has no validated candidates -- nothing to analyze yet. Check "
        ".symbro/pmpnn.csv, or loosen --max-rmsd/--min-plddt and rerun `symbro predict`."
    )

print(f"{len(predict_df)} validated candidate(s) across {predict_df['assembly_id'].nunique()} "
      f"assembly/assemblies.")
predict_df.head()

## 2. Join ProteinMPNN's own score back in

`predict.pkl` doesn't carry the amino acid sequence that produced each validated
candidate, or ProteinMPNN's own `global_score` -- only `candidate_id`/`folded_path`/
`rmsd_to_design`/`mean_plddt` (see `pipeline.py`'s own `_PREDICT_COLUMNS`). Both live in
`pmpnn.pkl` instead, joined back in here via `pipeline.join_predict_with_pmpnn()` --
the SAME helper `symbro codon` itself uses to recover the sequence it reverse-translates
(see `codon.py`), rather than a separate notebook-only reimplementation of the
`(assembly_id, component_id, candidate_id)` join (and the NaN-safe `component_id`
handling it needs -- see that function's own docstring for exactly why, and
`test_predict.py`/`test_codon.py` for the two real bugs that shape once had).

In [ ]:
merged = pipeline.join_predict_with_pmpnn(predict_df, pmpnn_df)

unmatched = merged["global_score"].isna().sum()
if unmatched:
    print(f"Warning: {unmatched}/{len(merged)} candidate(s) didn't match a ProteinMPNN row -- "
          f"likely pmpnn.pkl/predict.pkl are from different runs. Those rows will show blank "
          f"global_score below.")

# symmetry_type is an assembly-level attribute, carried on rfdiffusion.pkl -- not
# predict.pkl/pmpnn.pkl -- so it's joined in on (assembly_id, component_id) too.
def _component_key(df):
    # Same NaN-safe merge key join_predict_with_pmpnn() uses internally
    # (see pipeline.py's own _component_key()) -- component_id is None
    # for a single-component assembly, and a plain merge on a float NaN
    # column doesn't reliably match None to None.
    return df["component_id"].apply(lambda v: "__none__" if pd.isna(v) else str(v))

merged["_component_key"] = _component_key(merged)
rf_keyed = rfdiffusion_df.assign(_component_key=_component_key(rfdiffusion_df))
merged = merged.merge(
    rf_keyed[["assembly_id", "_component_key", "symmetry_type"]].drop_duplicates(),
    on=["assembly_id", "_component_key"], how="left",
).drop(columns="_component_key")

merged[["assembly_id", "component_id", "symmetry_type", "candidate_id", "rmsd_to_design",
        "mean_plddt", "global_score", "seq_recovery"]]

## 3. Self-consistency RMSD vs. pLDDT

Left: colored by assembly, so you can see at a glance whether one assembly is
consistently outperforming another. Right: the same points colored by ProteinMPNN's
`global_score` (lower = the sequence model was more confident in that candidate) — a
free secondary signal layered on top of the two metrics that actually decided pass/fail
at screening time.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for aid, group in merged.groupby("assembly_id"):
    ax.scatter(group["rmsd_to_design"], group["mean_plddt"], label=aid, s=70, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Self-consistency RMSD to design (Å)")
ax.set_ylabel("Mean pLDDT")
ax.set_title("Validated candidates, by assembly")
ax.legend(fontsize=8, title="assembly_id")

ax2 = axes[1]
sc = ax2.scatter(merged["rmsd_to_design"], merged["mean_plddt"], c=merged["global_score"],
                  cmap="viridis_r", s=70, edgecolor="black", linewidth=0.5)
ax2.set_xlabel("Self-consistency RMSD to design (Å)")
ax2.set_ylabel("Mean pLDDT")
ax2.set_title("Colored by ProteinMPNN global_score\n(lower = more confident)")
plt.colorbar(sc, ax=ax2, label="global_score")

plt.tight_layout()
plt.show()

## 4. Per-assembly / symmetry-type breakdown

How many validated candidates each assembly ended up with, and how good the best one
is. This is a breakdown of *validated* candidates only (see the limitation noted at the
top) — not a pass rate against everything that was attempted, since the attempted-but-
failed candidates were never saved.

In [ ]:
summary = merged.groupby(["assembly_id", "symmetry_type"], dropna=False).agg(
    validated_candidates=("candidate_id", "count"),
    best_rmsd=("rmsd_to_design", "min"),
    best_plddt=("mean_plddt", "max"),
    mean_plddt=("mean_plddt", "mean"),
).reset_index().sort_values("best_rmsd")

summary

## 5. Build a tighter shortlist (optional)

Every row here already passed whatever `--max-rmsd`/`--min-plddt` you ran `symbro
predict` with. If you want a stricter cutoff for picking a final synthesis shortlist
(rather than everything that merely passed screening), set it here — this only filters
the in-memory table below, it doesn't touch any checkpoint.

In [ ]:
RMSD_CUTOFF = 2.0    # angstrom -- same units/meaning as `symbro predict --max-rmsd`
PLDDT_CUTOFF = 70.0  # same units/meaning as `symbro predict --min-plddt`

within_cutoff = (merged["rmsd_to_design"] <= RMSD_CUTOFF) & (merged["mean_plddt"] >= PLDDT_CUTOFF)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(merged.loc[within_cutoff, "rmsd_to_design"], merged.loc[within_cutoff, "mean_plddt"],
           color="tab:green", label="within cutoff", s=70, edgecolor="black", linewidth=0.5)
ax.scatter(merged.loc[~within_cutoff, "rmsd_to_design"], merged.loc[~within_cutoff, "mean_plddt"],
           color="tab:gray", label="outside cutoff", s=70, edgecolor="black", linewidth=0.5)
ax.axvline(RMSD_CUTOFF, color="red", linestyle="--", linewidth=1)
ax.axhline(PLDDT_CUTOFF, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Self-consistency RMSD to design (Å)")
ax.set_ylabel("Mean pLDDT")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"{within_cutoff.sum()}/{len(merged)} candidate(s) within RMSD_CUTOFF/PLDDT_CUTOFF above.")

## 6. Shortlist, sorted

Sorted by RMSD ascending (tie-broken by pLDDT descending) — the most literal "best
self-consistency first" ordering. `global_score`/`seq_recovery` are shown alongside for
context, not folded into a single combined score here; if you want a compound ranking
later, this table is the place to add one.

In [ ]:
shortlist = merged[within_cutoff].sort_values(
    ["rmsd_to_design", "mean_plddt"], ascending=[True, False]
).reset_index(drop=True)

shortlist[["assembly_id", "component_id", "symmetry_type", "candidate_id", "rmsd_to_design",
           "mean_plddt", "global_score", "seq_recovery", "folded_path"]]

## 7. Preview a shortlisted structure (optional)

Needs `py3Dmol`/`ipywidgets` (`pip install py3Dmol ipywidgets` if you don't have them) —
skipped gracefully if they're not installed. Reads `folded_path` directly, so this only
works if you're running the notebook on the same machine (or the same mounted
filesystem) the structure predictor actually wrote those files to.

In [ ]:
try:
    import py3Dmol
    import ipywidgets as widgets
    from IPython.display import display as ipy_display
except ImportError:
    print("py3Dmol/ipywidgets not installed -- skipping preview. "
          "`pip install py3Dmol ipywidgets` to enable this cell.")
else:
    folded_paths = [p for p in shortlist["folded_path"] if os.path.exists(p)]
    missing = len(shortlist) - len(folded_paths)
    if missing:
        print(f"Note: {missing} shortlisted structure(s) not found on disk from here -- "
              f"skipped in the dropdown below.")

    if not folded_paths:
        print("No shortlisted structure files reachable from this machine to preview.")
    else:
        def show_structure(path):
            view = py3Dmol.view(width=600, height=400)
            with open(path) as f:
                view.addModel(f.read(), "cif" if path.endswith(".cif") else "pdb")
            view.setStyle({"cartoon": {"colorscheme": "chainHetatm"}})
            view.zoomTo()
            return view

        design_dropdown = widgets.Dropdown(options=folded_paths, description="Design:")
        preview_area = widgets.Output()

        def _on_change(change):
            preview_area.clear_output()
            with preview_area:
                show_structure(change["new"]).show()

        design_dropdown.observe(_on_change, names="value")
        ipy_display(design_dropdown, preview_area)
        with preview_area:
            show_structure(folded_paths[0]).show()

## 8. Export the shortlist

Copies the shortlist table (as CSV) plus every reachable folded structure into
`analysis_shortlist/` in the current directory.

In [ ]:
import shutil

OUT_DIR = "analysis_shortlist"
os.makedirs(OUT_DIR, exist_ok=True)

shortlist.drop(columns="sequence", errors="ignore").to_csv(
    os.path.join(OUT_DIR, "shortlist.csv"), index=False
)

copied = 0
for p in shortlist["folded_path"]:
    if os.path.exists(p):
        shutil.copy(p, os.path.join(OUT_DIR, os.path.basename(p)))
        copied += 1

print(f"Wrote shortlist.csv ({len(shortlist)} row(s)) + {copied} structure file(s) to {OUT_DIR}/.")

---

## Notes / possible next steps

- This notebook only sees candidates that already passed `symbro predict`'s own
  screening. If you want to see the full attempted distribution (including
  near-misses) to judge how close your screening thresholds are to the real cutoff,
  that needs a small change to `toolkit/selfconsistency.py` to persist
  `results_df` (the unfiltered set) somewhere before `select_validated_designs()`
  filters it -- not done here.
- `global_score` is shown as a secondary signal, not folded into ranking -- per
  ProteinMPNN's own author, it isn't validated as a standalone quality ranker.
- Physics-based filters some binder-design pipelines layer on top (Rosetta ddG, shape
  complementarity, packing/SAP score, radius of gyration, clash score) aren't computed
  here -- they need a Rosetta install SymBro doesn't currently depend on.